# pi0.5 Navigation Feasibility Report

This notebook is **not** a matched head-to-head benchmark between Nav2 and pi0.5. Nav2 is used
only to generate **300 expert demonstration episodes (100 per map)** that are turned into the
LeRobot dataset used to fine-tune the pi0.5 VLA policy. Once fine-tuned, pi0.5 is deployed on
the **same maps** and its behaviour is evaluated qualitatively: we look at whether its metrics
**trend towards** the Nav2 demonstrations it was trained on (a positive feasibility signal) or
diverge from them, using **per-episode metric tables, spatial trajectory overlays, and an
explicit trend-over-episodes plot**, instead of a single aggregate comparison curve — a
head-to-head statistic would be misleading here since the two controllers are not evaluated
under a symmetric, matched design.

## How the input data is produced

Run `eval_logger_node.py` (in `Mir250/mir_navigation/`) as a passive observer alongside
either controller — it only listens to `/episode_goal`, odometry and `cmd_vel`, so it does
not interfere with Nav2 or the pi0.5 inference node:

```bash
# While mir_random_nav.py (Nav2) drives: collect the 100 expert demonstrations per map
# (300 total) later used to fine-tune pi0.5.
python3 eval_logger_node.py --ros-args -p controller_name:=nav2 -p map_name:=maze

# While goal_monitor_node.py + inference_ros2_node.py (pi0.5) drive: log as many
# evaluation episodes as needed on the same maps to observe pi0.5's trend.
python3 eval_logger_node.py --ros-args -p controller_name:=pi05 -p map_name:=maze \
    -p cmd_vel_topic:=/diff_cont/cmd_vel_unstamped
```

Both runs append to the same `eval_metrics.csv` (default location `~/nav_eval_logs/`), with
per-episode trajectories saved under `~/nav_eval_logs/trajectories/`. Repeat for every map to
build up the dataset used below.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import yaml
from PIL import Image

pd.set_option("display.width", 120)

# ---- Configuration: adjust to match where eval_logger_node.py wrote its output ----
OUTPUT_DIR = Path.home() / "nav_eval_logs"
EVAL_CSV = OUTPUT_DIR / "eval_metrics.csv"
TRAJ_DIR = OUTPUT_DIR / "trajectories"

# Repo maps (world_name.yaml + world_name.pgm), used for the trajectory overlay background
MAPS_DIR = Path("../../../Maps")  # relative to Mir250/mir_navigation/analysis/

print(f"EVAL_CSV : {EVAL_CSV} (exists={EVAL_CSV.exists()})")
print(f"TRAJ_DIR : {TRAJ_DIR} (exists={TRAJ_DIR.exists()})")
print(f"MAPS_DIR : {MAPS_DIR.resolve()} (exists={MAPS_DIR.exists()})")


In [ ]:
## Load per-episode metrics
if not EVAL_CSV.exists():
    print(
        f"No eval_metrics.csv found at {EVAL_CSV}.\n"
        "Run eval_logger_node.py for both 'nav2' and 'pi05' controllers first "
        "(see instructions in the first cell)."
    )
    df = pd.DataFrame()
else:
    df = pd.read_csv(EVAL_CSV)
    print(f"Loaded {len(df)} episodes from {EVAL_CSV}")

df.head()


In [ ]:
## Aggregated comparison table (controller x map)
if df.empty:
    summary = pd.DataFrame()
else:
    def _agg(g: pd.DataFrame) -> pd.Series:
        n = len(g)
        failed = g[g["result"] != "success"]
        return pd.Series({
            "n_episodes": n,
            "success_rate_%": 100.0 * (g["success"].sum() / n),
            "timeout_rate_%": 100.0 * ((g["result"] == "timeout").sum() / n),
            "time_s_mean": g["time_s"].mean(),
            "time_s_std": g["time_s"].std(),
            "path_length_m_mean": g["path_length_m"].mean(),
            "path_efficiency_mean": g["path_efficiency"].mean(),
            "final_dist_m_mean_on_failure": failed["final_dist_m"].mean() if len(failed) else np.nan,
            "mean_abs_ang_vel": g["mean_abs_ang_vel"].mean(),
            "ang_vel_std_mean": g["ang_vel_std"].mean(),
        })

    rows = []
    for (controller, map_name), g in df.groupby(["controller", "map"]):
        row = _agg(g)
        row["controller"] = controller
        row["map"] = map_name
        rows.append(row)
    summary = pd.DataFrame(rows)
    cols = ["controller", "map"] + [c for c in summary.columns if c not in ("controller", "map")]
    summary = summary[cols].round(3)

summary


**Reading the table:** the `nav2` row(s) describe the expert demonstrations pi0.5 was
fine-tuned on — treat them as the **reference**, not as a competitor score. The `pi05` row(s)
are the evaluation-episode statistics gathered afterwards on the same maps. The most relevant
comparison is not "which controller wins" but **how close** `success_rate_%` and
`path_efficiency_mean` get to the Nav2 reference, and how large `final_dist_m_mean_on_failure`
and `ang_vel_std_mean` are — large/unstable values here indicate the transfer is not yet
feasible, while values approaching the Nav2 reference are a positive feasibility signal. The
next section plots this trend directly across pi0.5's evaluation episodes.


## pi0.5 trend across evaluation episodes (feasibility signal)

Since this is a feasibility study, the central question is not a single averaged number but
whether pi0.5's behaviour **trends towards** the Nav2 demonstrations as evaluation episodes
accumulate. The plots below show a rolling window of pi0.5's `success` and `path_efficiency`
across its own chronological `episode_id`, with the corresponding Nav2 reference value (mean
over the 100 demonstration episodes for that map) drawn as a dashed horizontal line.


In [ ]:
## pi0.5 trend plot vs Nav2 reference (rolling window over evaluation episodes)
if df.empty:
    print("No data loaded yet.")
else:
    ROLL_WINDOW = 10  # rolling window size (in episodes) used to smooth the trend

    for map_name in sorted(df["map"].unique()):
        pi05_ep = (
            df[(df["controller"] == "pi05") & (df["map"] == map_name)]
            .sort_values("episode_id")
            .reset_index(drop=True)
        )
        if pi05_ep.empty:
            continue

        nav2_ref = summary[(summary["controller"] == "nav2") & (summary["map"] == map_name)]
        nav2_success_ref = nav2_ref["success_rate_%"].values[0] if len(nav2_ref) else np.nan
        nav2_pe_ref = nav2_ref["path_efficiency_mean"].values[0] if len(nav2_ref) else np.nan

        fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

        rolling_success = pi05_ep["success"].rolling(ROLL_WINDOW, min_periods=1).mean() * 100.0
        axes[0].plot(pi05_ep["episode_id"], rolling_success, color="tab:orange",
                     label=f"pi0.5 (rolling {ROLL_WINDOW}-ep. success %)")
        if not np.isnan(nav2_success_ref):
            axes[0].axhline(nav2_success_ref, color="tab:blue", linestyle="--",
                             label="Nav2 demonstrations (reference)")
        axes[0].set_ylabel("success rate (%)")
        axes[0].set_title(f"pi0.5 evaluation trend on '{map_name}'")
        axes[0].legend(loc="lower right")

        rolling_pe = pi05_ep["path_efficiency"].rolling(ROLL_WINDOW, min_periods=1).mean()
        axes[1].plot(pi05_ep["episode_id"], rolling_pe, color="tab:orange",
                     label=f"pi0.5 (rolling {ROLL_WINDOW}-ep. path efficiency)")
        if not np.isnan(nav2_pe_ref):
            axes[1].axhline(nav2_pe_ref, color="tab:blue", linestyle="--",
                             label="Nav2 demonstrations (reference)")
        axes[1].set_ylabel("path efficiency")
        axes[1].set_xlabel("episode id (chronological)")
        axes[1].legend(loc="lower right")

        plt.tight_layout()
        plt.show()


In [ ]:
## Map loading helper (for the trajectory overlay background)
def load_map(map_name: str):
    """Return (image_array, resolution, origin_xy) for a ROS map yaml+pgm pair."""
    yaml_path = MAPS_DIR / f"{map_name}.yaml"
    if not yaml_path.exists():
        raise FileNotFoundError(f"Map yaml not found: {yaml_path.resolve()}")
    with open(yaml_path) as f:
        meta = yaml.safe_load(f)
    pgm_path = yaml_path.parent / meta["image"]
    img = np.array(Image.open(pgm_path).convert("L"))
    resolution = float(meta["resolution"])
    origin = meta["origin"]  # [x, y, yaw]
    return img, resolution, (float(origin[0]), float(origin[1]))


def world_to_pixel(x, y, img_h, resolution, origin_xy):
    """ROS map convention: pgm row 0 = top = map y-max."""
    ox, oy = origin_xy
    col = (x - ox) / resolution
    row = img_h - (y - oy) / resolution
    return col, row


In [ ]:
## Trajectory overlay plot: Nav2 vs pi0.5 on top of the map
COLORS = {"nav2": "tab:blue", "pi05": "tab:orange"}

def plot_trajectories(map_name: str, controllers=("nav2", "pi05")):
    img, resolution, origin_xy = load_map(map_name)
    img_h = img.shape[0]

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(img, cmap="gray", origin="upper")

    episodes = df[df["map"] == map_name]
    for controller in controllers:
        color = COLORS.get(controller, None)
        ep_rows = episodes[episodes["controller"] == controller]
        label_used = False
        for _, ep in ep_rows.iterrows():
            traj_path = TRAJ_DIR / ep["trajectory_file"]
            if not traj_path.exists():
                continue
            tdf = pd.read_csv(traj_path)
            px, py = world_to_pixel(tdf["x"].values, tdf["y"].values, img_h, resolution, origin_xy)
            style = "-" if ep["success"] == 1 else "--"
            ax.plot(px, py, style, color=color, linewidth=1.3, alpha=0.8,
                     label=controller if not label_used else None)
            label_used = True
            gx, gy = world_to_pixel(ep["goal_x"], ep["goal_y"], img_h, resolution, origin_xy)
            ax.plot(gx, gy, "*", color=color, markersize=10)
        sx, sy = world_to_pixel(0.0, 0.0, img_h, resolution, origin_xy)
    ax.plot(sx, sy, "ko", markersize=6, label="start")

    ax.set_title(f"Trajectories on '{map_name}' — solid = success, dashed = failure/timeout")
    ax.legend(loc="upper right")
    ax.set_xlabel("pixel (x)")
    ax.set_ylabel("pixel (y)")
    plt.tight_layout()
    plt.show()


In [ ]:
## Plot overlays for every map present in the metrics
if not df.empty:
    for map_name in sorted(df["map"].unique()):
        plot_trajectories(map_name)
else:
    print("No data loaded yet — run eval_logger_node.py first.")


## Qualitative case studies: pi0.5 failures

Aggregate numbers and the trend plot above show *whether* pi0.5's behaviour is approaching
feasibility; they don't explain *why* individual episodes fail. The cell below surfaces the
worst pi0.5 episodes (timed out, or finished furthest from the goal) so you can go back to the
corresponding RViz recording / screenshot and describe the failure mode (e.g. stuck against an
obstacle, drifted off in the wrong direction, ignored the prompt) — this is the main qualitative
evidence used to judge feasibility.


In [ ]:
if df.empty:
    print("No data loaded yet.")
else:
    failures = df[(df["controller"] == "pi05") & (df["success"] == 0)].copy()
    failures = failures.sort_values("final_dist_m", ascending=False)
    cols = ["episode_id", "map", "result", "time_s", "path_efficiency",
            "final_dist_m", "trajectory_file"]
    display(failures[cols].head(10))


## Summary

This workflow deliberately replaces a single aggregate comparison curve with:

1. **Descriptive tables** of the Nav2 expert demonstrations (success rate, timeout rate,
   time-to-goal, path efficiency, command smoothness) used as a reference, alongside pi0.5's
   own evaluation-episode statistics.
2. **A trend plot** of pi0.5's success rate and path efficiency across its chronological
   evaluation episodes, against the Nav2 reference — the central feasibility signal.
3. **Spatial trajectory overlays** on the actual map, showing *where* pi0.5's routes track or
   depart from the Nav2 demonstrations (solid = success, dashed = failure/timeout).
4. **Targeted qualitative case studies** of the worst pi0.5 episodes, pointing back to the
   original RViz recordings for manual failure-mode analysis.

This is more appropriate than a single aggregate statistic given that Nav2 here only supplies
training demonstrations (not a matched evaluation run), while pi0.5's feasibility as a
navigation policy is best judged through its trend over episodes and concrete failure cases
rather than a head-to-head score.
